# Пример submission

Минимальный пример структуры решения. Перед отправкой обязательно замените данные участника на свои и дополните notebook собственным анализом и пояснениями.


## Данные участника

ФИО: Борцов Михаил Алексеевич
Email: mialbortsov@edu.hse.ru
ВУЗ: ВШЭ


## Загрузка открытых данных

Скачайте Public Data из соревнования. В архиве находится `train.csv`. Истинные кластерные метки не публикуются, поскольку задача является кластеризацией.


In [ ]:
import pandas as pd, numpy as np
train = pd.read_csv("train.csv")
print(train.shape)
train.head()
print(train.describe().T)
print("Пропуски:", train.isna().sum().sum())

## Абоба

Ниже используется StandardScaler и KMeans. Число кластеров выбирается по silhouette score. Также показано, что `dataset_id` можно использовать для разных веток алгоритма.


In [ ]:
"""Improved clustering solution (better than solver/solution.py).

Improvements:
- NaN-safe input (drop all-NaN columns, median imputation);
- StandardScaler + PCA denoising (95% variance, <=8 dims);
- per-dataset k validated on public train data, refined at runtime
  only on a strong silhouette signal (avoids splitting rings into arcs);
- KMeans + Spectral + GMM ensemble with deterministic consensus scoring.

Usage: labels = cluster(X, dataset_id)   # deterministic, random_state=42
"""
import numpy as np
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, normalized_mutual_info_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

# k validated on public train data (see notebook.ipynb).
K_PRIOR = {
    "ds00": 2, "ds01": 3, "ds02": 2, "ds03": 2, "ds04": 3, "ds05": 3,
    "ds06": 3, "ds07": 3, "ds08": 2, "ds09": 3, "ds10": 3, "ds11": 4,
    "ds12": 2, "ds13": 2, "ds14": 2, "ds15": 4, "ds16": 2, "ds17": 2,
    "ds18": 2, "ds19": 4, "ds20": 4, "ds21": 3, "ds22": 2, "ds23": 2,
}


def _clean(X):
    X = np.asarray(X, dtype=float)
    keep = ~np.all(np.isnan(X), axis=0)
    X = X[:, keep] if keep.any() else np.zeros((len(X), 1))
    for j in range(X.shape[1]):
        col = X[:, j]
        bad = np.isnan(col)
        if bad.any():
            X[bad, j] = np.nanmean(col)
    return X


def _reduce(X):
    Z = StandardScaler().fit_transform(X)
    Z = np.where(np.isfinite(Z), Z, 0.0)
    if X.shape[1] > 3:
        p = PCA(random_state=42).fit(Z)
        cum = np.cumsum(p.explained_variance_ratio_)
        n = max(2, min(8, int(np.searchsorted(cum, 0.95) + 1), X.shape[1]))
        Z = PCA(n_components=n, random_state=42).fit_transform(Z)
    return Z


def _sil(Z, lab):
    try:
        u = np.unique(lab)
        if len(u) < 2 or len(u) >= len(Z):
            return -1.0
        return float(silhouette_score(Z, lab))
    except Exception:
        return -1.0


def _consensus(cands):
    out = []
    for i, a in enumerate(cands):
        vals = [normalized_mutual_info_score(a, b)
                for j, b in enumerate(cands) if j != i]
        out.append(float(np.mean(vals)) if vals else 0.0)
    return np.array(out)


def cluster(X, dataset_id=None):
    """Return labels only (grading API). Use explain() for the full report."""
    return _run(X, dataset_id)[0]


def explain(X, dataset_id=None):
    """Same pipeline as cluster(); returns dict with 'labels' plus which of the
    KMeans / SpectralClustering / GaussianMixture candidates won and why.
    Keys: labels, k_prior, k_used, sil_by_k, methods {name: score}, winner."""
    labels, info = _run(X, dataset_id)
    out = {'labels': labels}
    out.update(info)
    return out


def _run(X, dataset_id=None):
    Xc = _clean(X)
    n = len(Xc)
    if n < 4:
        lab = np.zeros(n, dtype=int)
        info = {'k_prior': None, 'k_used': 1, 'sil_by_k': {},
                'methods': {'constant': 0.0}, 'winner': 'constant'}
        return lab, info
    Z = _reduce(Xc)

    ds = str(dataset_id or "").lower().strip("'\" ")
    k0 = K_PRIOR.get(ds, 3)
    ks = sorted({max(2, k0 - 1), k0, min(k0 + 1, max(2, n // 5))})

    # --- choose k: prior first; override only on a strong signal ---
    parts, sils = {}, {}
    for k in ks:
        parts[k] = KMeans(n_clusters=k, n_init=15, random_state=42).fit(Z).labels_
        sils[k] = _sil(Z, parts[k])
    best_k = max(ks, key=lambda k: -sils[k])
    if best_k != k0 and sils[best_k] - sils[k0] < 0.04:
        best_k = k0  # e.g. keep ring+center as 2, don't split the ring
    best_k = min(best_k, max(2, n // 3))

    # --- ensemble at best_k ---
    cands = [('KMeans', parts[best_k])]
    try:
        sp = SpectralClustering(n_clusters=best_k, affinity="nearest_neighbors",
                                n_neighbors=min(n - 1, max(10, int(np.sqrt(n)))),
                                random_state=42).fit_predict(Z)
        if len(np.unique(sp)) == best_k:
            cands.append(('SpectralClustering', sp))
    except Exception:
        pass
    try:
        gm = GaussianMixture(n_components=best_k, covariance_type="full",
                             n_init=3, random_state=42).fit(Z)
        lab_gm = gm.predict(Z)
        if len(np.unique(lab_gm)) == best_k:
            cands.append(('GaussianMixture', lab_gm))
    except Exception:
        pass

    score = np.array([_sil(Z, c) for _, c in cands]) + 0.3 * _consensus([c for _, c in cands])
    i = int(np.argmax(score))
    labels = np.asarray(cands[i][1], dtype=int)
    info = {
        'k_prior': k0,
        'k_used': best_k,
        'sil_by_k': {k: round(float(sils[k]), 3) for k in ks},
        'methods': {name: round(float(s), 3) for (name, _), s in zip(cands, score)},
        'winner': cands[i][0],
    }
    return labels, info

## Выводы

KMeans является только отправной точкой. Для улучшения Mean ARI необходимо исследовать свойства отдельных dataset_id, предобработку, альтернативные алгоритмы кластеризации и гиперпараметры.
